# Transformer-Based Token-Level Detection of Human–Machine Text Boundaries
SemEval-2024 Task 8, Subtask C. Goal: find the **word index where the machine-written text starts**.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom.

Stages: 1 Dataset · 2 Preprocessing · 3 Tokenization · 4 RoBERTa model + training · 5 Token predictions · 6 Boundary reconstruction · 7 Evaluation · 8 Demo

## Stage 1 — Dataset input

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [25]:
import random, numpy as np, pandas as pd, torch
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          get_linear_schedule_with_warmup)

# ---- settings (change only here) ----
SEED       = 42
MODEL_NAME = "roberta-base"
MAX_LEN    = 512      # RoBERTa limit
BATCH      = 8
EPOCHS     = 3
LR         = 2e-5

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", device)
if device.type == "cpu":
    print("WARNING: no GPU found. Training on CPU will take many hours. Switch to a T4 GPU.")

Device: cuda


In [26]:
ds = load_dataset("d0rj/SemEval2024-task8", "subtaskC")
train_df = ds["train"].to_pandas()
dev_df   = ds["dev"].to_pandas()
test_df  = ds["test"].to_pandas()

for name, d in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    d["word_count"] = d["text"].str.split().str.len()   # SAME split() as the labelling
    print(name, d.shape, "| missing:", int(d.isnull().sum().sum()))

print(train_df["label"].describe())

train (3649, 4) | missing: 0
dev (505, 4) | missing: 0
test (11123, 4) | missing: 0
count    3649.000000
mean       71.456289
std        71.515904
min         0.000000
25%        22.000000
50%        51.000000
75%        99.000000
max       755.000000
Name: label, dtype: float64


In [27]:
# Data sanity check (we do NOT delete anything, we only look)
for name, d in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    n_zero  = int((d["label"] == 0).sum())
    n_over  = int((d["label"] >= d["word_count"]).sum())
    print(f"{name}: boundary = 0 -> {n_zero} | boundary >= word_count -> {n_over}")

train: boundary = 0 -> 232 | boundary >= word_count -> 2
dev: boundary = 0 -> 23 | boundary >= word_count -> 0
test: boundary = 0 -> 371 | boundary >= word_count -> 55


## Stage 2 — Preprocessing (word-level labels)
Rule: `word_index < boundary` → HUMAN (0), otherwise MACHINE (1).
We keep boundary = 0 samples (all MACHINE). No samples are removed.

In [ ]:
def word_labels(n_words, boundary):
    return [0 if i < boundary else 1 for i in range(n_words)]

# quick check on the first training sample
w = train_df.loc[0, "text"].split()
b = int(train_df.loc[0, "label"])
lab = word_labels(len(w), b)
print("boundary:", b)
for i in range(b - 3, b + 3):
    print(i, w[i], "HUMAN" if lab[i] == 0 else "MACHINE")

tot_h = sum(int(l) for l in train_df["label"].clip(upper=train_df["word_count"]))
tot_w = int(train_df["word_count"].sum())
print("HUMAN %:", round(100 * tot_h / tot_w, 1), "| MACHINE %:", round(100 * (tot_w - tot_h) / tot_w, 1))

boundary: 88
85 Also HUMAN
86 its HUMAN
87 not HUMAN
88 clear MACHINE
89 to MACHINE
90 me MACHINE
HUMAN %: 30.0 | MACHINE %: 70.0


## Stage 3 — Tokenization and label alignment
**Correction:** `add_prefix_space=True` is needed for RoBERTa when the text is given as a list of words. Without it every word is tokenized as if it were in the middle of a word, which does not match how RoBERTa was pretrained.

**Simplification:** we keep only the first 512 tokens (no sliding windows). In your data only 35 of 3,649 boundaries fall after token 512.

In [28]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)

def encode_example(text, boundary=None):
    words = text.split()
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    word_ids = enc.word_ids()
    item = {"input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "word_ids": word_ids,
            "n_words": len(words)}
    if boundary is not None:
        item["labels"] = [-100 if w is None else (0 if w < boundary else 1) for w in word_ids]
    return item

def build_items(df):
    return [encode_example(t, int(b)) for t, b in tqdm(zip(df["text"], df["label"]), total=len(df))]

train_items = build_items(train_df)
dev_items   = build_items(dev_df)
print("train items:", len(train_items), "| dev items:", len(dev_items))

  0%|          | 0/3649 [00:00<?, ?it/s]

  0%|          | 0/505 [00:00<?, ?it/s]

train items: 3649 | dev items: 505


In [ ]:
# Alignment checks
bad = sum(len(x["input_ids"]) != len(x["labels"]) for x in train_items + dev_items)
print("Length mismatches:", bad)

# the tokens must now look like normal RoBERTa tokens (with the G-dot space marker)
ex = train_items[0]
toks = tokenizer.convert_ids_to_tokens(ex["input_ids"])
print(toks[:12])

# how many samples lose their boundary because of truncation?
def cut_off(items, df):
    n = 0
    for it, b in zip(items, df["label"]):
        seen = max([w for w in it["word_ids"] if w is not None], default=-1) + 1
        if seen < it["n_words"] and int(b) >= seen:
            n += 1
    return n
print("train boundaries beyond 512 tokens:", cut_off(train_items, train_df))
print("dev   boundaries beyond 512 tokens:", cut_off(dev_items, dev_df))

Length mismatches: 0
['<s>', 'Ġ-', 'ĠStre', 'ng', 'ths', ':', 'Ġ*', 'ĠOut', 'per', 'forms', 'ĠAL', 'IGN']
train boundaries beyond 512 tokens: 16
dev   boundaries beyond 512 tokens: 2


In [29]:
# Batching: pad each batch to its longest sequence (padding gets label -100 -> ignored by the loss)
def collate(batch):
    L = max(len(x["input_ids"]) for x in batch)
    ids  = torch.full((len(batch), L), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    has_labels = "labels" in batch[0]
    labs = torch.full((len(batch), L), -100, dtype=torch.long) if has_labels else None
    for i, x in enumerate(batch):
        n = len(x["input_ids"])
        ids[i, :n]  = torch.tensor(x["input_ids"])
        mask[i, :n] = torch.tensor(x["attention_mask"])
        if has_labels:
            labs[i, :n] = torch.tensor(x["labels"])
    out = {"input_ids": ids, "attention_mask": mask}
    if has_labels:
        out["labels"] = labs
    return out

train_loader = DataLoader(train_items, batch_size=BATCH, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_items,   batch_size=16,    shuffle=False, collate_fn=collate)
print("train batches:", len(train_loader), "| dev batches:", len(dev_loader))

train batches: 457 | dev batches: 32


## Stage 4 — RoBERTa token-classification model and fine-tuning
Pretrained `roberta-base` + a new 2-class head (HUMAN / MACHINE). Loss ignores `-100` labels automatically.
We train 3 epochs, check the dev set after every epoch, and keep the best model.

In [30]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/roberta_boundary_model_v2"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "HUMAN", 1: "MACHINE"}, label2id={"HUMAN": 0, "MACHINE": 1})
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate_tokens(loader):
    model.eval()
    tot_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out = model(**batch)
            tot_loss += out.loss.item()
            pred = out.logits.argmax(-1)
            m = batch["labels"] != -100
            correct += (pred[m] == batch["labels"][m]).sum().item()
            count += m.sum().item()
    return tot_loss / len(loader), correct / count

# reference: accuracy of always predicting the majority class on dev
dev_lab = np.concatenate([[l for l in x["labels"] if l != -100] for x in dev_items])
print("Dev share of MACHINE tokens:", round(float((dev_lab == 1).mean()), 3),
      "(a model that always says MACHINE would get about this accuracy)")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dev share of MACHINE tokens: 0.684 (a model that always says MACHINE would get about this accuracy)


In [ ]:
best_dev_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(**batch)
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += out.loss.item()

    dev_loss, dev_acc = evaluate_tokens(dev_loader)
    print(f"Epoch {epoch}: train loss {running/len(train_loader):.4f} | "
          f"dev loss {dev_loss:.4f} | dev token accuracy {dev_acc:.4f}")

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print("  -> best model so far, saved to Drive")

Epoch 1/3:   0%|          | 0/457 [00:00<?, ?it/s]

/tmp/ipykernel_2162/3158553487.py:16: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1: train loss 0.1807 | dev loss 0.1246 | dev token accuracy 0.9566


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> best model so far, saved to Drive


Epoch 2/3:   0%|          | 0/457 [00:00<?, ?it/s]

... (3 outputs left out)

Epoch 3/3:   0%|          | 0/457 [00:00<?, ?it/s]

Epoch 3: train loss 0.0447 | dev loss 0.0664 | dev token accuracy 0.9779


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> best model so far, saved to Drive


In [31]:
# Load the best saved model for everything that follows
model = AutoModelForTokenClassification.from_pretrained(SAVE_DIR).to(device)
model.eval()
if 'best_dev_loss' in globals():
    print("Best model loaded. Dev loss:", round(best_dev_loss, 4))
else:
    print("Best model loaded. (Dev loss variable not found in current session)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Best model loaded. (Dev loss variable not found in current session)


**Coming back later?** Do not retrain. Run Stage 1 and Stage 3, **skip the model-creation and training cells in Stage 4**, then run this cell instead.

## Stage 5 — Token-level predictions → word-level probabilities
For every word we average P(MACHINE) over its subword tokens. This avoids the old rule of using only the first subword.

In [32]:
@torch.no_grad()
def predict_word_probs(items, batch_size=32, show_progress=True):
    model.eval()
    order = np.argsort([len(x["input_ids"]) for x in items])   # similar lengths together = faster
    results = [None] * len(items)
    starts = range(0, len(items), batch_size)
    for s in (tqdm(starts) if show_progress else starts):
        idx = order[s:s + batch_size]
        batch = collate([items[i] for i in idx])
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
        probs = torch.softmax(logits.float(), dim=-1)[:, :, 1].cpu().numpy()   # P(MACHINE)
        for row, i in enumerate(idx):
            wids = items[i]["word_ids"]
            n_seen = max([w for w in wids if w is not None], default=-1) + 1
            sums, cnt = np.zeros(n_seen), np.zeros(n_seen)
            for pos, w in enumerate(wids):
                if w is not None:
                    sums[w] += probs[row, pos]
                    cnt[w] += 1
            results[i] = sums / np.maximum(cnt, 1)
    return results

dev_probs = predict_word_probs(dev_items)
print("Dev texts predicted:", len(dev_probs), "(must be 505)")
print("Example, first 10 word probabilities of dev text 0:", np.round(dev_probs[0][:10], 2))

  0%|          | 0/16 [00:00<?, ?it/s]

Dev texts predicted: 505 (must be 505)
Example, first 10 word probabilities of dev text 0: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## Stage 6 — Boundary reconstruction (best single cut)
The old rule "first word predicted MACHINE" breaks if one early word is wrong.
Better: try every possible cut `b` and pick the one that disagrees least with the model:

`cost(b) = sum of P(MACHINE) of words before b  +  sum of P(HUMAN) of words from b on`

The answer is the `b` with the lowest cost. This always gives exactly one HUMAN → MACHINE switch, as the task requires.

In [33]:
def decode_boundary(word_probs, total_words):
    p = np.asarray(word_probs, dtype=float)
    n = len(p)
    if n == 0:
        return 0
    c = np.concatenate([[0.0], np.cumsum(p)])      # c[b] = sum of p[:b]
    b = np.arange(n + 1)
    cost = c[b] + (n - b) - (c[n] - c[b])
    return int(min(np.argmin(cost), total_words))

# tiny self-test: 3 clear HUMAN words, then 4 clear MACHINE words -> boundary must be 3
print("self-test:", decode_boundary([0.1, 0.05, 0.1, 0.9, 0.95, 0.9, 0.99], 7))

self-test: 3


## Stage 7 — Evaluation (MAE)
MAE = average of |predicted boundary − true boundary|. Lower is better.
Use the dev set to decide anything. Run the test cell **only once**, at the very end.

In [ ]:
def evaluate(probs, df, name):
    pred = np.array([decode_boundary(p, int(n)) for p, n in zip(probs, df["word_count"])])
    gold = df["label"].astype(int).to_numpy()
    err = np.abs(pred - gold)
    base = int(train_df["label"].median())
    base_mae = mean_absolute_error(gold, [base] * len(gold))
    print(f"{name} MAE            : {err.mean():.2f}")
    print(f"{name} median error   : {np.median(err):.1f} words")
    print(f"{name} constant-guess MAE (always {base}): {base_mae:.2f}")
    for k in (5, 10, 25):
        print(f"{name} within {k:>2} words: {(err <= k).mean()*100:.1f}%")
    return pred, gold, err

dev_pred, dev_gold, dev_err = evaluate(dev_probs, dev_df, "DEV")

DEV MAE            : 4.64
DEV median error   : 2.0 words
DEV constant-guess MAE (always 51): 45.47
DEV within  5 words: 73.5%
DEV within 10 words: 87.7%
DEV within 25 words: 97.6%


In [ ]:
# Error analysis by where the true boundary is
bins = pd.cut(dev_gold, [-1, 0, 50, 100, 200, 10000],
              labels=["0", "1-50", "51-100", "101-200", ">200"])
print(pd.DataFrame({"bucket": bins, "error": dev_err}).groupby("bucket", observed=True)["error"]
        .agg(["count", "mean", "median"]).round(1))

for i in range(8):
    print("Actual:", dev_gold[i], "| Predicted:", dev_pred[i], "| Error:", dev_err[i])

         count  mean  median
bucket                      
0           23   0.0     0.0
1-50       234   3.1     1.0
51-100     131   4.5     2.0
101-200     94   6.6     4.0
>200        23  18.0    11.0
Actual: 94 | Predicted: 93 | Error: 1
Actual: 86 | Predicted: 93 | Error: 7
Actual: 403 | Predicted: 381 | Error: 22
Actual: 12 | Predicted: 14 | Error: 2
Actual: 75 | Predicted: 67 | Error: 8
Actual: 42 | Predicted: 44 | Error: 2
Actual: 53 | Predicted: 55 | Error: 2
Actual: 261 | Predicted: 256 | Error: 5


In [ ]:
# RUN ONLY ONCE, after you are happy with the dev results. Do not tune on this.
test_items = [encode_example(t) for t in tqdm(test_df["text"])]
test_probs = predict_word_probs(test_items)
test_pred, test_gold, test_err = evaluate(test_probs, test_df, "TEST")

print()
print("Reported in the base paper (Puspo et al., 2024), NOT our results:")
print("  Weighted ensemble : dev 31.71 | test 60.78")
print("  Longformer baseline: dev about 3.53 | test 21.54")

  0%|          | 0/11123 [00:00<?, ?it/s]

  0%|          | 0/348 [00:00<?, ?it/s]

TEST MAE            : 17.91
TEST median error   : 7.0 words
TEST constant-guess MAE (always 51): 56.78
TEST within  5 words: 45.3%
TEST within 10 words: 60.8%
TEST within 25 words: 79.8%

Reported in the base paper (Puspo et al., 2024), NOT our results:
  Weighted ensemble : dev 31.71 | test 60.78
  Longformer baseline: dev about 3.53 | test 21.54


## Stage 8 — Demo inside Colab (no web link needed)
Run **8a** first (helper), then **8b** (three ready-made demos), then **8c** (your own text box).

Notes: the model was trained on peer-review-style text, and the task assumes **human text first, machine text after**. Use at least 20 words (100+ words works best).

In [34]:
# 8a — helper: runs the model and shows the result with colours
import html
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def show_result(text, title="Result", expected=None):
    words = text.split()
    if len(words) < 20:
        display(HTML("<b>Please enter at least 20 words.</b>")); return
    item = encode_example(text)
    probs = predict_word_probs([item], show_progress=False)[0]
    seen = len(probs)                                   # words the model could read (first 512 tokens)
    b = decode_boundary(probs, len(words))

    def span(ws, bg):
        return f"<span style='background:{bg};color:#000;padding:2px 3px;border-radius:3px'>{html.escape(' '.join(ws))}</span> "
    h = span(words[:b], "#c8f0d2") if b > 0 else ""
    m = span(words[b:seen], "#f8cccc") if b < seen else ""
    r = span(words[seen:], "#dddddd") if seen < len(words) else ""

    if b == 0:
        verdict = "The whole text looks MACHINE-generated (boundary = 0)."
    elif b >= seen and seen == len(words):
        verdict = "No machine text found: the whole text looks HUMAN."
    elif b >= seen:
        verdict = "No switch found in the first 512 tokens (grey part was not analysed)."
    else:
        verdict = f"Machine text starts at word index <b>{b}</b> (text has {len(words)} words)."
    extra = f" &nbsp;|&nbsp; expected boundary: <b>{expected}</b>" if expected is not None else ""
    display(HTML(
        f"<h4>{html.escape(title)}</h4>"
        f"<p><span style='background:#c8f0d2;color:#000;padding:2px 6px'>HUMAN</span> "
        f"<span style='background:#f8cccc;color:#000;padding:2px 6px'>MACHINE</span> "
        f"<span style='background:#dddddd;color:#000;padding:2px 6px'>not analysed</span></p>"
        f"<div style='line-height:2'>{h}{m}{r}</div>"
        f"<p>{verdict}{extra}</p>"
        f"<p style='font-size:90%'>Average P(machine) over the text: {np.mean(probs):.2f}</p><hr>"))

In [41]:
# 8b — three ready-made demos (texts written by hand for this demo)
human_text = (
"""Once I settled down at the Schwartz High School,Ramanathapuram, the enthusiastic fifteen-year-old within me re-emerged.My teacher,
Iyadurai Solomon,was an ideal guide for an eager young mind that was yet uncertain of the possibilities and alternatives that lay
before it.He made his students feel very comfortable in class with his warm and open-minded attitude.He used to say
that a good student could learn more form a bad teacher than a poor student form even a skilled teacher.At St Joseph's, I was lucky
to find a teacher like the Rev. Father T. N. Sequeira. He taught us English and was also our hostel warden. We were about a hundred
boys living in the three-storeyed hostel building. Rev. Father used to visit each boy every night with a Bible in his hand. His energy
and patience was amazing. He was a very considerate person who took care of even the most minute requirements of his students. On Deepavali,
on his instructions, the Brother in charge of the hostel and the mess volunteers would visit each room and distribute good gingelly oil for the ritual bath.
""")

show_result(human_text, "Demo 1 — fully human-written text (expected: no machine text)", expected="none (= number of words)")

In [43]:
ai_text = (
"""During my four years living on the St. Joseph's campus, I shared a room with two companions—an orthodox Iyengar from Srirangam and a
Syrian Christian from Kerala. Together, we built a wonderful camaraderie. In my third year, after taking on the role of secretary for the
vegetarian mess, we invited our Rector, Rev. Father Kalathil, to join us for Sunday lunch. We crafted a menu featuring the finest traditional
dishes from each of our diverse cultural backgrounds. Though the culinary combination turned out to be quite unexpected, Father Kalathil praised
our initiative lavishly and joined in our simple, candid conversations with genuine, childlike enthusiasm, making the afternoon truly unforgettable.
The faculty at St. Joseph's truly embodied the teachings of the Kanchi Paramacharya, who urged people to find deep joy in the act of giving. To this
day, I remain profoundly inspired by the enduring memory of our mathematics professors, Prof. Thothathri Iyengar and Prof. Suryanarayana Sastry,
walking side by side across the campus grounds.As I entered my final year at St. Joseph's, I discovered an appreciation for English literature.
I immersed myself in classic works—developing a particular fondness for Tolstoy, Scott, and Hardy despite their unfamiliar, foreign settings—before
turning my attention toward philosophical writings. It was during this phase of intellectual exploration that a profound passion for Physics began to take root.""")
show_result(ai_text,    "Demo 2 — fully AI-written text (expected: boundary 0)", expected=0)


In [37]:
human_part = (
"""Once I settled down at the Schwartz High School,Ramanathapuram, the enthusiastic fifteen-year-old within me re-emerged.My teacher,
Iyadurai Solomon,was an ideal guide for an eager young mind that was yet uncertain of the possibilities and alternatives that lay
before it.He made his students feel very comfortable in class with his warm and open-minded attitude.He used to say
that a good student could learn more form a bad teacher than a poor student form even a skilled teacher.""")
machine_part = (
"""During my tenure in Ramanathapuram, my relationship with him evolved beyond the traditional scope of a teacher and student dynamic.
Through his guidance, I realized that an individual possesses substantial agency to exert influence over the trajectory of their own life.
Iyadurai Solomon was accustomed to stating, 'To achieve success and yield meaningful results in life, one must comprehend and master three
fundamental imperatives—desire, conviction, and expectation.'Iyadurai Solomon, who subsequently assumed the office of a Reverend, imparted
the principle that prior to the realization of any objective, one must cultivate an intense desire alongside an unwavering certainty regarding
its occurrence.To illustrate this principle through an autobiographical account, I had been captivated by atmospheric phenomena and avian flight
from early childhood. I frequently observed the soaring flight of cranes and seagulls, nurturing a profound ambition to fly. Despite my modest,
provincial origins, I remained steadfast in the conviction that I would eventually ascend into the skies. Ultimately, I became the first native
of Rameswaram to achieve flight.""")
mixed_text = human_part + " " + machine_part

show_result(mixed_text, "Demo 2 — human first, AI next", expected=len(human_part.split()))


In [ ]:
def scores_near_boundary(df, idx, margin=10):
    row = df.iloc[idx]
    words = row["text"].split()
    true_b = int(row["label"])
    item = encode_example(row["text"])
    probs = predict_word_probs([item], show_progress=False)[0]
    pred_b = decode_boundary(probs, len(words))

    print(f"Sample {idx} | true boundary {true_b} | predicted {pred_b} | error {abs(pred_b - true_b)}")
    lo, hi = max(0, true_b - margin), min(len(probs), true_b + margin)
    print(f"{'idx':>4}  {'word':<15} {'P(machine)':>10}  final")
    for i in range(lo, hi):
        print(f"{i:>4}  {words[i]:<15} {probs[i]:>10.3f}  {'MACHINE' if i >= pred_b else 'HUMAN'}")

for i in [0, 1, 2, 3, 4]:          # first 5 training samples
    scores_near_boundary(train_df, i)
    print()

Sample 0 | true boundary 88 | predicted 94 | error 6
 idx  word            P(machine)  final
  78  dimension            0.000  HUMAN
  79  of                   0.000  HUMAN
  80  500                  0.000  HUMAN
  81  vs                   0.000  HUMAN
  82  this                 0.000  HUMAN
  83  paper's              0.000  HUMAN
  84  200.                 0.000  HUMAN
  85  Also                 0.002  HUMAN
  86  its                  0.019  HUMAN
  87  not                  0.068  HUMAN
  88  clear                0.091  HUMAN
  89  to                   0.069  HUMAN
  90  me                   0.047  HUMAN
  91  whether              0.178  HUMAN
  92  N(e_j)               0.260  HUMAN
  93  includes             0.396  HUMAN
  94  only                 0.653  MACHINE
  95  entities             0.608  MACHINE
  96  that                 0.777  MACHINE
  97  link                 0.728  MACHINE

Sample 1 | true boundary 286 | predicted 279 | error 7
 idx  word            P(machine)  final
 27

In [ ]:
N = 300
sub = train_df.iloc[:N]
probs_list = predict_word_probs(train_items[:N])
pred = np.array([decode_boundary(p, int(n)) for p, n in zip(probs_list, sub["word_count"])])
gold = sub["label"].astype(int).to_numpy()
err = np.abs(pred - gold)

print("TRAIN (first 300) MAE:", round(err.mean(), 2))
print("TRAIN median error  :", np.median(err))
print("TRAIN within 5 words:", round((err <= 5).mean() * 100, 1), "%")

  0%|          | 0/10 [00:00<?, ?it/s]

TRAIN (first 300) MAE: 5.1
TRAIN median error  : 3.0
TRAIN within 5 words: 71.7 %


In [ ]:
# 8c — your own text: type or paste, then click Detect
box = widgets.Textarea(
    placeholder="Paste your text here (at least 20 words, 100+ works best)...",
    layout=widgets.Layout(width="95%", height="220px"))
button = widgets.Button(description="Detect boundary", button_style="primary")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        show_result(box.value, "Your text")

button.on_click(on_click)
display(box, button, out)

Textarea(value='', layout=Layout(height='220px', width='95%'), placeholder='Paste your text here (at least 20 …

Button(button_style='primary', description='Detect boundary', style=ButtonStyle())

Output()